In [1]:
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    
    !uv pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130    
    !uv pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --extra-index-url https://download.pytorch.org/whl/cu130 --index-strategy unsafe-best-match
    !uv pip install unsloth_zoo bitsandbytes accelerate xformers peft --extra-index-url https://download.pytorch.org/whl/cu130 --index-strategy unsafe-best-match
    !uv pip install wandb
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !uv pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !uv pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
!uv pip install --upgrade torchao --extra-index-url https://download.pytorch.org/whl/cu130 --index-strategy unsafe-best-match

Checked 2 packages in 71ms
   Updating https://github.com/unslothai/unsloth.git (HEAD)
    Updated https://github.com/unslothai/unsloth.git (b53cc957dcc43bb105b0fde3ac42c92458eeaba5)
Resolved 83 packages in 20.86s
   Building unsloth @ git+https://github.com/unslothai/unsloth.git@b53cc957dcc43bb105b0fde3ac42c92458eeaba5
      Built unsloth @ git+https://github.com/unslothai/unsloth.git@b53cc957dcc43bb105b0fde3ac42c92458eeaba5
Prepared 1 package in 56.43s
Uninstalled 1 package in 1.04s
Installed 1 package in 2.79s
 - unsloth==2026.6.7 (from git+https://github.com/unslothai/unsloth.git@9d7740a82f2ec2649c74d95154e91f63c376b3ef)
 + unsloth==2026.6.7 (from git+https://github.com/unslothai/unsloth.git@b53cc957dcc43bb105b0fde3ac42c92458eeaba5)
Checked 5 packages in 24ms
Checked 1 package in 21ms
Checked 1 package in 13ms
Checked 1 package in 18ms
Resolved 1 package in 1.28s
Checked 1 package in 0.26ms


In [2]:
import gc
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig
import torch
from datasets import load_dataset
import wandb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextStreamer
from huggingface_hub import list_models, login
import logging
import warnings

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logging.captureWarnings(True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0615 23:37:43.547000 27936 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


In [3]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU Detected: {gpu_stats.name}. Max memory = {max_memory} GB.")
else:
    print("No GPU detected.")

GPU Detected: NVIDIA GeForce RTX 5050 Laptop GPU. Max memory = 7.96 GB.


In [5]:
print("\nModel Unsloth Qwen3 tersedia:")
for m in list_models(author="unsloth", search="qwen3", sort="downloads", limit=25):
    print(f"  {m.id}")


Model Unsloth Qwen3 tersedia:
  unsloth/Qwen3-Coder-Next-GGUF
  unsloth/Qwen3.6-35B-A3B-GGUF
  unsloth/Qwen3.6-27B-GGUF
  unsloth/Qwen3.6-27B-MTP-GGUF
  unsloth/Qwen3.6-35B-A3B-MTP-GGUF
  unsloth/Qwen3.5-9B-GGUF
  unsloth/Qwen3-4B
  unsloth/Qwen3-VL-2B-Instruct-GGUF
  unsloth/Qwen3.5-4B-GGUF
  unsloth/Qwen3-30B-A3B-Instruct-2507-GGUF
  unsloth/Qwen3.6-27B-NVFP4
  unsloth/Qwen3.5-0.8B-GGUF
  unsloth/Qwen3-14B-unsloth-bnb-4bit
  unsloth/Qwen3.5-9B-MTP-GGUF
  unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF
  unsloth/Qwen3.5-9B
  unsloth/Qwen3-0.6B
  unsloth/Qwen3.6-35B-A3B-NVFP4
  unsloth/Qwen3.5-35B-A3B-GGUF
  unsloth/Qwen3.5-2B-GGUF
  unsloth/Qwen3.5-27B-GGUF
  unsloth/Qwen3-4B-unsloth-bnb-4bit
  unsloth/Qwen3.5-4B
  unsloth/Qwen3.5-4B-MTP-GGUF
  unsloth/Qwen3-8B-unsloth-bnb-4bit


In [6]:
model_name = "unsloth/Qwen3-4B"
max_seq_len = 1024
max_train_len = 1024
system_prompt = "Kamu adalah asisten AI yang membantu menjawab pertanyaan pengguna berdasarkan data dan fakta yang kamu miliki."

In [7]:
temp_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
temp_tokenizer = get_chat_template(temp_tokenizer, chat_template="qwen2.5")

Model does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [8]:
ds = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")
# Membagi dataset
ds = ds.train_test_split(test_size=0.1, seed=3407)

In [ ]:
ds["train"] = ds["train"].shuffle(seed=3407).select(range(40000))
ds["test"] = ds["test"].shuffle(seed=3407).select(range(200))
print(f"Jumlah data latih: {len(ds['train'])}")
print(f"Jumlah data evaluasi: {len(ds['test'])}")
print(ds["train"][100])

Jumlah data latih: 40000
Jumlah data evaluasi: 200


In [10]:
def formatting_prompts_func(examples, tokenizer):
    
    inputs  = examples["input"]
    outputs = examples["output"]
    texts = []

    for user_input, output in zip(inputs, outputs):
        conversation = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input.strip()},
            {"role": "assistant", "content": output.strip()}
        ]
        
        # 2. Gunakan argumen 'tokenizer', bukan 'temp_tokenizer' global
        text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        texts.append(text)
        
    return { "text" : texts }

# 3. Jalankan mapping dengan fn_kwargs untuk meneruskan tokenizer
ds = ds.map(
    formatting_prompts_func,
    batched=True,
    fn_kwargs={"tokenizer": temp_tokenizer}
)

In [11]:
ds["train"][100]["text"]

'<|im_start|>system\nKamu adalah asisten AI yang membantu menjawab pertanyaan pengguna berdasarkan data dan fakta yang kamu miliki.<|im_end|>\n<|im_start|>user\nDesain chatbot untuk website layanan pelanggan yang dibangun dengan GPT-3.<|im_end|>\n<|im_start|>assistant\nChatbot layanan pelanggan yang ditenagai oleh GPT-3 akan menjadi alat yang sangat bagus untuk memberikan dukungan instan, akurat, dan personal untuk pelanggan pada sebuah website. Berikut adalah desain yang mungkin untuk chatbot tersebut:\n\n1. Pesan sambutan: Ketika pelanggan memasuki website, jendela chat akan muncul, menampilkan pesan sambutan, dan menawarkan bantuan. Contohnya: "Halo! Selamat datang di website kami. Saya adalah asisten kecerdasan buatan yang ditenagai oleh GPT-3, dan saya di sini untuk membantu Anda dengan segala pertanyaan yang Anda miliki."\n\n2. Pertanyaan pengguna: Pelanggan kemudian dapat mengetik atau mengucapkan pertanyaan, kekhawatiran, atau masalah mereka pada jendela chat. Pesan tersebut ak

In [12]:
del temp_tokenizer
gc.collect()
torch.cuda.empty_cache()

In [13]:
USE_WANDB = True
if USE_WANDB:
    os.environ["WANDB_API_KEY"] = os.getenv('WANDB_TOKEN')
    wandb.init(project="Qwen3-Finetuning", name="Unsloth_Run")
    report_target = "wandb"
else:
    os.environ["WANDB_DISABLED"] = "true"
    report_target = None

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rizkyayub (rizkyayub-freelancer) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


In [14]:
# ── 9. Training Config & Eksperimen Loop ──────────────────────────────────────
experiments = [
    {"lr": 1e-5, "run_name": "Run_1_LR_1e-5", "gradient_accumulation_steps": 4, "warmup_steps": 50},
    {"lr": 2e-5, "run_name": "Run_2_LR_2e-5", "gradient_accumulation_steps": 8, "warmup_steps": 70}
]


for exp in experiments:
    print(f"\n\n{'='*50}\nMemulai eksperimen: {exp['run_name']}\n{'='*50}")

    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_len,
        load_in_4bit = True,
        load_in_8bit = False,
        full_finetuning = False,
        max_lora_rank = 8,
        dtype = None

    )    
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "qwen2.5"
    )

    model = FastLanguageModel.get_peft_model(
        model,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
        r = 8,
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing="unsloth",
        random_state = 3407
    )
    
    sft_config = SFTConfig(
        output_dir = f"outputs_{exp['run_name']}",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = exp["gradient_accumulation_steps"],
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        max_length = max_train_len,
        warmup_steps = exp["warmup_steps"],
        max_steps = 800,
        learning_rate = exp["lr"],
        lr_scheduler_type = "cosine",
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        optim = "paged_adamw_8bit",
        logging_steps = 10,
        dataset_text_field = "text",
        per_device_eval_batch_size = 2,
        eval_strategy = "steps",
        eval_steps = 200,
        save_steps = 200,
        save_total_limit = 2,
        ddp_find_unused_parameters = False,
        dataloader_num_workers = 0,
        report_to = report_target,
        run_name = exp["run_name"]
    )

    trainer = SFTTrainer(
        model = model,
        processing_class = tokenizer,
        train_dataset = ds["train"],
        eval_dataset = ds["test"],
        args = sft_config,
        packing = False
    )

    trainer.train()

    del trainer
    gc.collect()
    torch.cuda.empty_cache() # Flush memori setelah training selesai
    torch.cuda.synchronize()# Sinkronisasi GPU untuk memastikan semua operasi selesai sebelum menyimpan model

    model.save_pretrained(f"lora_model_{exp['run_name']}")
    tokenizer.save_pretrained(f"lora_model_{exp['run_name']}")

    if exp != experiments[-1]: # Jangan hapus model jika ini adalah eksperimen terakhir (untuk inferensi di bawah)
        del model
        gc.collect()
        torch.cuda.empty_cache()
        
        
if USE_WANDB:
    wandb.finish()



Memulai eksperimen: Run_1_LR_1e-5
==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 5050 Laptop GPU. Num GPUs = 1. Max memory: 7.96 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.6.7 patched 36 layers with 36 QKV layers, 36 O layers and 0 MLP layers.


Unsloth: Tokenizing ["text"]:   0%|          | 0/200 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 40,000 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 5,898,240 of 4,028,366,336 (0.15% trained)


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 40,000 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 5,898,240 of 4,028,366,336 (0.15% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
200,1.153800,1.151167
400,1.167100,1.103241
600,1.152600,1.093723
800,1.118500,1.092231


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/loss,█▄▃▃▄▂▁▁
eval/runtime,▃▂▁▃▇█▄▄
eval/samples_per_second,▆▇█▆▂▁▅▅
eval/steps_per_second,▆▇█▆▂▁▅▅
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▁▁▂▂▃▃▄▄▄▅▅▇▇███
train/global_step,▁▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██▁▁▁▂▂▂▄▄▅▅▅▅▆▆▇▇▇█
train/grad_norm,▃▃▄█▂▁▂▂▂▂▃▂▂▂▂▃▃▂▂▂▂▂▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂
train/learning_rate,▅▄▄▄▄▄▄▃▃▃▂▂▂▂▂▂▁▁▂▃████▇▇▆▆▆▆▆▄▄▃▃▂▂▁▁▁
train/loss,▇▇▄▂▂▂▂▂▂▁▂▁▁▂▂▂▇▇█▂▁▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,1.09223
eval/runtime,34.7601


In [15]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560, padding_idx=151669)
        (layers): ModuleList(
          (0-1): 2 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
  

In [16]:
terminators = []
if isinstance(tokenizer.eos_token_id, list):
    terminators.extend(tokenizer.eos_token_id)
elif tokenizer.eos_token_id is not None:
    terminators.append(tokenizer.eos_token_id)
if not terminators:
    terminators = [tokenizer.eos_token_id or 128001]
pad_id = tokenizer.pad_token_id or terminators[0]

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Sebutkan 3 bandara terkenal di Indonesia beserta lokasinya!"}
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)
attention_mask = torch.ones_like(inputs)
text_streamer = TextStreamer(tokenizer, skip_prompt=True)  # ← pindah ke sini

_ = model.generate(
    input_ids          = inputs,
    attention_mask     = attention_mask,
    streamer           = text_streamer,
    max_new_tokens     = 256,
    temperature        = 0.1,        # ← dari Kode 1
    top_k              = 20,          # ← dari Kode 1
    top_p              = 0.9,        # ← dari Kode 1
    repetition_penalty = 1.05,        # ← dari Kode 1, lebih aman untuk model 4B
    do_sample          = True,
    eos_token_id       = terminators, # ← safe list dari Kode 2
    pad_token_id       = pad_id,      # ← safe fallback dari Kode 2
)

1. Bandara Soekarno-Hatta, Jakarta - Bandara ini adalah bandara utama di Jakarta dan merupakan pusat penerbangan internasional terbesar di Indonesia.
2. Bandara Halim Perdanakusuma, Jakarta - Bandara ini adalah bandara kedua di Jakarta dan merupakan pusat penerbangan internasional yang sangat penting bagi kota tersebut.
3. Bandara Ngurah Rai, Bali - Bandara ini adalah bandara utama di Bali dan merupakan pusat penerbangan internasional yang sangat penting bagi pulau tersebut.<|im_end|>


In [ ]:
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN tidak ditemukan!"
login(token=HF_TOKEN)

hf_username = "rizkyayub"
running_model = experiments[-1]["run_name"]
repo_id = f"{hf_username}/rizbuy-submission-qwen3-indonesian"

print(f"\nMengunggah model '{running_model}' ke HuggingFace {repo_id}")

torch.cuda.empty_cache()
torch.cuda.synchronize()

try:
    model.push_to_hub_merged(
        repo_id = repo_id,
        tokenizer = tokenizer,
        save_method = "merged_16bit",
        token = HF_TOKEN,
        private = False,
        commit_message = f"Fine-tuned Qwen3-4B Indonesian - {running_model}"
    )
    print(f"✅ Model berhasil diunggah ke: https://huggingface.co/{repo_id}")
except Exception as e:
    print(f"Upload gagal: {e}")
    model.save_pretrained(f"lora_model_backup_{running_model}")
    tokenizer.save_pretrained(f"lora_model_backup_{running_model}")
    print("Model disimpan lokal sebagai fallback.")


Mengunggah model 'Run_2_LR_2e-5' ke HuggingFace rizkyayub/rizbuy-submission-qwen.3-indonesian


No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


...

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [02:08<02:08, 128.20s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [06:32<00:00, 196.03s/it]


Unsloth: Merge process complete. Saved to `C:\Users\Rizky\AppData\Local\Temp\tmp425nb9yc`
✅ Model berhasil diunggah ke: https://huggingface.co/rizkyayub/rizbuy-submission-qwen.3-indonesian
